In [ ]:
print(1)

In [ ]:
import anndata as ad
import pandas as pd
import numpy as np
import os
import pickle
import torch
from transformers import AutoTokenizer, AutoModel

In [ ]:
ROOT_OUTPUT_PATH = 'data_coverage_match'
ROOT_INPUT_PATH = 'data_coverage_match'

PATH = f'../{ROOT_OUTPUT_PATH}/benchmark/resources/datasets/neurips-2023-data-subsample'
os.makedirs(PATH, exist_ok=True)

# Helpers

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from rdkit.DataStructs import ConvertToNumpyArray

def smiles_to_fingerprints(smiles_list, radius=1, fp_size=2000):
    """Convert a list of SMILES strings to Morgan (ECFP) fingerprint matrix.

    Args:
        smiles_list: Iterable of SMILES strings.
        radius: Morgan fingerprint radius (default 1, i.e. ECFP2).
        fp_size: Fingerprint bit vector size.

    Returns:
        np.ndarray of shape (len(smiles_list), fp_size), dtype uint8.
        Invalid SMILES yield a zero vector for that row.
    """
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=fp_size)
    fps = []
    for smile in smiles_list:
        mol = Chem.MolFromSmiles(smile)
        if mol is None:
            fps.append(None)
        else:
            fp = gen.GetFingerprint(mol)
            arr = np.zeros(fp_size, dtype=np.uint8)
            ConvertToNumpyArray(fp, arr)
            fps.append(arr)
    return fps

In [ ]:

def smiles_to_emb(smiles, version=1):
    if version == 1:
        name = "DeepChem/ChemBERTa-77M-MLM"
    elif version == 2:
        name = "seyonec/ChemBERTa-zinc250k-v1"
    elif version == 3:
        name = "ibm-research/MoLFormer-XL-both-10pct"
    else:
        raise ValueError(f"unknown version={version}")

    if version == 3:
        tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
        model = AutoModel.from_pretrained(
            name, deterministic_eval=True, trust_remote_code=True
        ).eval()
    else:
        tok = AutoTokenizer.from_pretrained(name)
        model = AutoModel.from_pretrained(name).eval()

    enc = tok(smiles, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        out = model(**enc)

    if version == 3:
        print(out.pooler_output.shape)  # (batch, 768)
        return out.pooler_output.tolist()

    h = out.last_hidden_state  # (batch, seq_len, hidden)  hidden=384 v1 / 768 v2
    print(h.shape)
    # masked mean pooling (ignores padding) -- good default
    mask = enc["attention_mask"].unsqueeze(-1)
    emb = (h * mask).sum(1) / mask.sum(1)  # (batch, hidden)
    return emb.tolist()

# 1. Split

In [ ]:
op3_train_subsample = ad.read_h5ad(f'../{ROOT_INPUT_PATH}/benchmark/resources/datasets/neurips-2023-data/de_train.h5ad')
op3_test_subsample = ad.read_h5ad(f'../{ROOT_INPUT_PATH}/benchmark/resources/datasets/neurips-2023-data/de_test.h5ad')
df = pd.read_csv(f'../{ROOT_INPUT_PATH}/benchmark/resources/datasets/neurips-2023-data/id_map.csv')

In [ ]:
op3_test_subsample

In [ ]:
op3_subsample = ad.concat([op3_train_subsample, op3_test_subsample], uns_merge='same')

In [ ]:
df_single_cell_obs = pd.concat([op3_train_subsample.uns['single_cell_obs'], op3_test_subsample.uns['single_cell_obs']])

In [ ]:
ratio = 0.25
compounds = np.array(op3_subsample.obs['sm_name'].unique())
np.random.seed(42)
test_sample = np.random.choice(compounds, size=int(len(compounds) * ratio), replace=False)
train_compounds = set(compounds) - set(test_sample)

In [ ]:
op3_subsample.obs['new_split'] = np.where(op3_subsample.obs['sm_name'].isin(test_sample), 'test', 'train')

In [ ]:
op3_train_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'train'].copy()
op3_train_subsample_.uns['single_cell_obs'] = df_single_cell_obs[~df_single_cell_obs['sm_name'].isin(test_sample)]
op3_train_subsample_.write_h5ad(f'{PATH}/de_train.h5ad', compression='gzip')

In [ ]:
op3_test_subsample_ = op3_subsample[op3_subsample.obs['new_split'] == 'test'].copy()
op3_test_subsample_.uns['single_cell_obs'] = df_single_cell_obs[df_single_cell_obs['sm_name'].isin(test_sample)]
op3_test_subsample_.write_h5ad(f'{PATH}/de_test.h5ad', compression='gzip')

In [ ]:
op3_test_subsample_.obs[['sm_name', 'cell_type']].reset_index(drop=True).reset_index().rename(columns={'index': 'id'}).to_csv(f'{PATH}/id_map.csv', index=False)

In [ ]:
assert len(set(op3_test_subsample_.obs['sm_name']).intersection(set(op3_train_subsample_.obs['sm_name']))) == 0

In [ ]:
len(op3_test_subsample_.obs['sm_name'].unique()) + len(op3_train_subsample_.obs['sm_name'].unique())

# 2. Get Fingerprints

In [ ]:
sm_smiles = op3_subsample.obs[['sm_name', 'SMILES']].drop_duplicates()
sm_smiles['ECFP:2'] = smiles_to_fingerprints(sm_smiles['SMILES'])

sm_smiles['ChemBERTa-v1'] = smiles_to_emb(sm_smiles['SMILES'].tolist())
sm_smiles['ChemBERTa-v2'] = smiles_to_emb(sm_smiles['SMILES'].tolist(), version=2)
sm_smiles['MoLFormer-XL'] = smiles_to_emb(sm_smiles['SMILES'].tolist(), version=3)

sm_smiles = sm_smiles.rename(columns = {'SMILES': 'smiles', 'sm_name': 'perturbagen'})

In [ ]:
sm_smiles

# 3. Load Pubchem

In [ ]:
#PubChemCIDs were obtained with the Chem-PerturBridge pipeline with the PubChemPy package
df_emb_op3 = pd.read_csv('../files/df_pubchem_op3.csv')
df_emb_op3['pubchem_cid'] = df_emb_op3['pubchem_cid'].astype(str)

# 4. Get embeddings and JOIN

In [ ]:
epoch_dirs = ['epoch_epoch_0019']

In [ ]:
def load_and_join(checkpoint_file):
    """Join one LPM checkpoint with OP3 compounds and chemical features."""
    df_pert = pd.read_pickle(checkpoint_file).rename(
        columns={'lpm_style_embeddings': 'LPM_emb'}
    )
    return (
        df_emb_op3.merge(df_pert, left_on='pubchem_cid', right_on='symbol', how='left')
        .merge(sm_smiles, on='perturbagen', how='left')
    )


def save_embedding(df, filename, feature='ECFP:2', require_lpm=True):
    """Keep the required features and coverage-matched compounds, then save."""
    required = [feature]
    if require_lpm:
        required.append('LPM_emb')

    columns = ['perturbagen', 'LPM_emb', 'smiles', feature]
    result = df.dropna(subset=required)[columns].reset_index(drop=True)
    result = result[result['perturbagen'].isin(allowed_compounds)]
    result.to_pickle(f'{PATH}/{filename}')

In [ ]:
reference_checkpoint = '../files/single_run_all_data_l1000_narrowed/epoch_epoch_0019/df_pert.pkl'
reference_embeddings = load_and_join(reference_checkpoint).dropna(subset=['LPM_emb', 'ECFP:2'])

covered_test_compounds = set(test_sample) & set(reference_embeddings['perturbagen'])
allowed_compounds = train_compounds | covered_test_compounds

print(f'Retained {len(covered_test_compounds)} of {len(test_sample)} test compounds.')

## 4.1 All

In [ ]:
for epoch_dir in epoch_dirs:
    epoch_label = int(epoch_dir.split('_')[-1]) + 1
    joined = load_and_join(f'../files/single_run_all_25_epochs/{epoch_dir}/df_pert.pkl')
    save_embedding(joined, f'op3_emb_all_{epoch_label}.pkl')

## 4.2 All (narrowed)

In [ ]:
for epoch_dir in epoch_dirs:
    epoch_label = int(epoch_dir.split('_')[-1]) + 1
    joined = load_and_join(f'../files/single_run_all_data_l1000_narrowed/{epoch_dir}/df_pert.pkl')
    save_embedding(joined, f'op3_emb_all_{epoch_label}_narrowed.pkl')

## 4.3 L1000

In [ ]:
for epoch_dir in epoch_dirs:
    epoch_label = int(epoch_dir.split('_')[-1]) + 1
    joined = load_and_join(f'../files/single_run_l1000_25_epochs/{epoch_dir}/df_pert.pkl')
    save_embedding(joined, f'op3_emb_l1000_{epoch_label}.pkl')

## 4.4 Chemical baselines

In [ ]:
baseline_checkpoint = '../files/single_run_all_25_epochs/epoch_epoch_0019/df_pert.pkl'
baseline_df = load_and_join(baseline_checkpoint)

baseline_features = {
    'fp': 'ECFP:2',
    'chemb1': 'ChemBERTa-v1',
    'chemb2': 'ChemBERTa-v2',
    'molformer': 'MoLFormer-XL',
}

for name, feature in baseline_features.items():
    save_embedding(baseline_df, f'op3_emb_{name}.pkl', feature=feature, require_lpm=False)

## 4.5 Ablation & stepmatched

In [ ]:
ablation_and_stepmatched_runs = [
    {
        'folder': 'single_run_abl_all_data_dataset_dose_time',
        'checkpoint': 'epoch_epoch_0019',
        'output': 'op3_emb_all_20_data_dataset_dose_time_coverage_match.pkl',
    },
    {
        'folder': 'single_run_abl_all_data_dataset',
        'checkpoint': 'epoch_epoch_0019',
        'output': 'op3_emb_all_20_data_dataset_coverage_match.pkl',
    },
    {
        'folder': 'single_run_abl_all_data_dose_time',
        'checkpoint': 'epoch_epoch_0019',
        'output': 'op3_emb_all_20_data_dose_time_coverage_match.pkl',
    },
    {
        'folder': 'single_run_abl_l1000_dataset_dose_time',
        'checkpoint': 'epoch_epoch_0019',
        'output': 'op3_emb_l1000_20_dataset_dose_time.pkl',
    },
    {
        'folder': 'single_run_abl_l1000_dataset',
        'checkpoint': 'epoch_epoch_0019',
        'output': 'op3_emb_l1000_20_dataset.pkl',
    },
    {
        'folder': 'single_run_abl_l1000_dose_time',
        'checkpoint': 'epoch_epoch_0019',
        'output': 'op3_emb_l1000_20_dose_time.pkl',
    },
    {
        'folder': 'single_run_all_data_stepmatched',
        'checkpoint': 'last',
        'output': 'op3_emb_all_last_stepmatched_coverage_match.pkl',
    },
    {
        'folder': 'single_run_l1000_data_stepmatched',
        'checkpoint': 'last',
        'output': 'op3_emb_l1000_last_stepmatched.pkl',
    },
]

In [ ]:
for run in ablation_and_stepmatched_runs:
    checkpoint_file = f"../files/{run['folder']}/{run['checkpoint']}/df_pert.pkl"
    joined = load_and_join(checkpoint_file)
    save_embedding(joined, run['output'])
    print(f"Saved to {PATH}/{run['output']}")